# m06 lab · Triangle meshes, Laplace–Beltrami operators and geodesics

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Maps live on triangle meshes. Conduction velocity needs gradients on triangles, smoothing and interpolation need a
Laplacian, and "distance from the earliest site" needs geodesics. You build each operator, test it on surfaces whose
answers are known (flat patches and spheres), and see which discretizations converge and which do not.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Meshes: counts, the Euler characteristic, triangle quality | eq-m06-euler |
| B | Gradients on triangles and conduction velocity from activation times | eq-m06-gradient |
| C | The cotangent Laplacian: flat checks, the sphere, negative weights | eq-m06-cotan |
| D | The spectrum of the sphere | eq-m06-spectrum |
| E | Distances: edge paths, the lattice formula and the heat method | eq-m06-geodesic |

Distances in mm on flat patches, relative on unit spheres. Each part ends with **checks**. No downloads; an Octave
copy of parts A–E is in `labs/octave/m06_lab.m`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import meshes as M
rng = np.random.default_rng(6)

## Part A · Meshes

A closed triangle mesh of genus $g$ satisfies $V - E + F = 2 - 2g$, and every edge is shared by two triangles, so
$E = 3F/2$ (eq-m06-euler). Build icospheres, count, and look at triangle quality: the smallest angle in each triangle,
which matters for every operator below.

In [ ]:
rows = []
for level in range(5):
    V, F = M.icosphere(level)
    rows.append((level, len(V), len(M.edges(F)), len(F), M.euler_characteristic(V, F)))
    print(f"level {level}: V = {rows[-1][1]:5d}, E = {rows[-1][2]:5d}, F = {rows[-1][3]:5d}, V - E + F = {rows[-1][4]}")
V, F = M.icosphere(3)
Vj, Fj = M.lattice_mesh(20, 1.0, jitter=0.35, seed=6)
min_sphere = np.degrees(M.face_angles(V, F).min(axis=1))
min_jit = np.degrees(M.face_angles(Vj, Fj).min(axis=1))
plt.hist(min_sphere, 30, alpha=0.7, label="icosphere, level 3"); plt.hist(min_jit, 30, alpha=0.7, label="jittered flat patch")
plt.xlabel("smallest angle in each triangle (degrees)"); plt.legend(); plt.show()
print(f"smallest angle: icosphere {min_sphere.min():.1f} degrees, jittered patch {min_jit.min():.1f} degrees")

In [ ]:
# Checks
assert all(chi == 2 and 2 * E == 3 * Fn for _, _, E, Fn, chi in rows)
assert min_sphere.min() > 50 and min_jit.min() < min_sphere.min()

## Part B · Gradients and conduction velocity

A function given at vertices is linear on each triangle; its gradient there is
$\nabla u = \frac{1}{2A}\sum_i u_i\,(\mathbf N \times \mathbf e_i)$ (eq-m06-gradient). From activation times $T$, the
conduction-velocity vector is $\nabla T/\lvert\nabla T\rvert^2$. It is exact for a planar wave on any triangulation.
Then add 1 ms of timing noise and change the point spacing: since speed is one over the gradient, small spacings
turn timing noise into large velocity errors.

In [ ]:
V, F = M.lattice_mesh(15, 1.0, jitter=0.25, seed=3)
d = np.array([np.cos(0.7), np.sin(0.7), 0.0])
lat = V @ d / 0.8                                     # a planar wave at 0.8 mm/ms (0.8 m/s)
v = M.conduction_velocity(V, F, lat)
speed_err = np.max(np.abs(np.linalg.norm(v, axis=1) - 0.8))
print(f"planar wave: largest speed error {speed_err:.1e} mm/ms")
spreads = {}
for h in (1.0, 2.0, 4.0, 8.0):
    Vh, Fh = M.lattice_mesh(15, h, jitter=0.25, seed=3)
    lat = Vh @ d / 0.8 + rng.normal(0, 1.0, len(Vh))   # 1 ms of timing noise
    s = np.linalg.norm(M.conduction_velocity(Vh, Fh, lat), axis=1)
    spreads[h] = np.median(np.abs(s - 0.8) / 0.8)
    print(f"spacing {h} mm with 1 ms noise: median speed error {100 * spreads[h]:.0f}%")

In [ ]:
# Checks
assert speed_err < 1e-12
assert spreads[1.0] > spreads[2.0] > spreads[4.0] > spreads[8.0]

Dense points do not by themselves give better velocity: the timing error has to shrink with the spacing. m25
treats velocity estimation properly.

## Part C · The cotangent Laplacian

The Laplace–Beltrami operator on a mesh is $(\Delta u)_i \approx \frac{1}{A_i}\sum_j \frac{\cot\alpha_{ij} +
\cot\beta_{ij}}{2}(u_j - u_i)$, with $\alpha_{ij}$, $\beta_{ij}$ the angles opposite edge $ij$ and $A_i$ one third of
the incident triangles' area (eq-m06-cotan; Pinkall and Polthier 1993). Check it on a flat patch and on the unit
sphere, where $\Delta z = -2z$, and count negative edge weights.

In [ ]:
Vj, Fj = M.lattice_mesh(15, 1.0, jitter=0.2, seed=1)
inner = (Vj[:, 0] > 2) & (Vj[:, 0] < 10) & (Vj[:, 1] > 2) & (Vj[:, 1] < 10)
lin_err = np.max(np.abs(M.laplace_beltrami(Vj, Fj, 2 * Vj[:, 0] - 3 * Vj[:, 1] + 1)[inner]))
Vr, Fr = M.lattice_mesh(15, 1.0)
inner_r = (Vr[:, 0] > 2) & (Vr[:, 0] < 10) & (Vr[:, 1] > 2) & (Vr[:, 1] < 10)
quad = M.laplace_beltrami(Vr, Fr, Vr[:, 0] ** 2 + Vr[:, 1] ** 2)[inner_r]
quad_j = M.laplace_beltrami(Vj, Fj, Vj[:, 0] ** 2 + Vj[:, 1] ** 2)[inner]
print(f"flat: linear function {lin_err:.1e}; x^2 + y^2 on the regular lattice {quad.min():.6f}-{quad.max():.6f}, "
      f"on the jittered patch {quad_j.min():.2f}-{quad_j.max():.2f} (exact 4)")
sph = []
for level in (2, 3, 4, 5):
    V, F = M.icosphere(level)
    e = np.abs(M.laplace_beltrami(V, F, V[:, 2]) + 2 * V[:, 2])
    sph.append((level, np.median(e), e.max()))
    print(f"sphere level {level}: median error {sph[-1][1]:.1e}, worst error {sph[-1][2]:.2f}")
Vk, Fk = M.lattice_mesh(20, 1.0, jitter=0.35, seed=6)      # the more distorted patch of part A
W = M.cotan_matrix(Vk, Fk).tocoo()
neg = np.sum((W.row != W.col) & (W.data < -1e-12)) // 2
print(f"strongly jittered patch: {neg} of {len(M.edges(Fk))} edges have a negative weight (not Delaunay)")

In [ ]:
# Checks
assert lin_err < 1e-10 and np.allclose(quad, 4.0)
assert sph[-1][1] < sph[0][1] / 30                      # typical error converges
assert sph[-1][2] > 0.2                                  # worst error, at valence-5 vertices, does not
assert neg > 0

Two cautions. The worst pointwise error, at the twelve vertices with five neighbors, does not shrink with barycentric
areas, even though the typical error does. And a jittered mesh has edges with negative weights, which can break
the maximum principle that smoothing and interpolation rely on (demo m06-d2).

## Part D · The spectrum of the sphere

The Laplace–Beltrami eigenvalues of a sphere of radius $R$ are $l(l + 1)/R^2$, each with multiplicity $2l + 1$
(eq-m06-spectrum). Solve $-L_C\varphi = \lambda A\varphi$ on refined icospheres.

In [ ]:
R = 2.0
spec = {}
for level in (2, 3, 4):
    V, F = M.icosphere(level, R)
    spec[level] = M.spectrum(V, F, k=16) * R ** 2
    print(f"level {level}:", np.round(spec[level], 3))
print("exact:   0, 2 (x3), 6 (x5), 12 (x7)")

In [ ]:
# Checks
assert np.allclose(spec[4][1:4], 2, rtol=0.01) and np.allclose(spec[4][4:9], 6, rtol=0.01) and np.allclose(spec[4][9:16], 12, rtol=0.02)
assert abs(spec[4][12] - 12) < abs(spec[2][12] - 12)

## Part E · Distances on a mesh

Distance along edges (Dijkstra) is easy but biased: on an equilateral lattice the shortest edge path is longer than
the straight line by exactly $(\sin\theta + \sin(60° - \theta))/\sin 60°$, up to $2/\sqrt3$ (15.5%), however fine the
mesh. The heat method (Crane et al. 2012) diffuses heat briefly, normalizes its gradient and solves a Poisson problem;
fast marching solves the eikonal equation $\lvert\nabla\varphi\rvert = 1$ on triangles (Kimmel and Sethian 1998). Both
converge (eq-m06-geodesic). Compare on the lattice and on spheres.

In [ ]:
V, F = M.lattice_mesh(41, 1.0)
src = 20 * 41 + 20
dist = np.linalg.norm(V - V[src], axis=1)
gd = M.graph_distance(V, F, src)
far = dist > 5
ang = np.arctan2(V[:, 1] - V[src, 1], V[:, 0] - V[src, 0])
lattice_err = np.max(np.abs(gd[far] / dist[far] - M.lattice_path_ratio(ang[far])))
print(f"lattice: edge-path ratio up to {np.max(gd[far] / dist[far]):.4f} (2/sqrt 3 = {2 / np.sqrt(3):.4f}); formula error {lattice_err:.1e}")
res = []
for level in (2, 3, 4, 5):
    V, F = M.icosphere(level)
    ex = M.sphere_distance(V, 0, 1.0)
    res.append((level, np.max(M.graph_distance(V, F, 0) - ex), np.mean(np.abs(M.heat_geodesic(V, F, 0) - ex))))
    print(f"sphere level {level}: edge-path excess up to {res[-1][1]:.3f}, heat-method mean error {res[-1][2]:.4f}")
plt.loglog([2 ** -r[0] for r in res], [r[1] for r in res], "o-", label="edge paths (worst)")
plt.loglog([2 ** -r[0] for r in res], [r[2] for r in res], "s-", label="heat method (mean)")
plt.xlabel("relative edge length"); plt.ylabel("distance error (unit sphere)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert lattice_err < 1e-12
assert res[-1][1] > 0.9 * res[0][1]                      # edge paths do not converge
assert res[-1][2] < res[0][2] / 4                        # the heat method does

## Where to go next

- In part B, keep the timing noise at 1 ms and find the spacing at which the median speed error falls below 10%.
- In part C, flip the non-Delaunay edges of the jittered patch. Do the negative weights disappear?
- In part E, try the heat method with $t = 10h^2$. What happens near the source and far from it?
- m20 builds geometry from catheter points, m24 interpolates on meshes, and m25 estimates conduction velocity.